In [1]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE" # File locking can cause long hangs on open/write.

import h5py # Read and write h5 files
from astropy.table import Table
from astropy.io import fits # Read fits files
import fitsio # -----------  "              "
from astropy.wcs import WCS # World Coordinate System for identifying centers of galaxies 
                            # (rushed solution to RA and DEC being group RA and DEC
import numpy as np
import pandas as pd
from pathlib import Path
import time # For efficiency metrics (obsolete if running small sets)
from tqdm.notebook import tqdm

from concurrent.futures import ProcessPoolExecutor, as_completed # Allow for concurrent work
from collections import deque # Improved efficiency at adding or removing info to the beginning of a list

In [2]:
# ============================================================
# USER CONFIGURATION
# ============================================================ 
output_dir = Path("/pscratch/sd/q/qshimp/Sorter")
anchor_path = Path("/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_4000_sga152x152_complete.csv")
name = "sga2025_sample_test3"

npix = 152
batch_size = 64
chunk_size = 1000
workers = 16  # ProcessPoolExecutor x16 -- best config from h5py_efficiency.ipynb (528.3 gal/s)
verbose = 4

# IMPORTANT: CHANGE TO MATCH YOUR DATA
anchor_columns = {
    # Change  -  Don't change
    "ra":       "target_ra",
    "dec":      "target_dec",
    "Main_type":"Main_Type",
    "ref_id":   "targetid",
    "Path":     "Path"
}

In [3]:
def prepare_output_directory(output_dir, name):  # Create output directory structure
    run_path = output_dir / "binary_classifier" / name
    run_path.mkdir(parents=True, exist_ok=True)
    return run_path
 
 
def read_catalog(path):  # Read in original data or anchors
    path = Path(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix == ".fits":
        return Table.read(path, hdu=1).to_pandas()
    raise ValueError(f"Unsupported file type: {path.suffix}")
 
 
def format_data(anchors, anchor_columns):
    anchor_df = read_catalog(anchors)
    anchor_df = (
        anchor_df[list(anchor_columns.keys())]
        .rename(columns=anchor_columns)
    return anchor_df
 
 
def get_band(bands, priority):
    for band in priority:
        if band in bands:
            return bands[band]
    return None

def crop_at_radec(data, header, ra, dec, npix):
    """Crop an npix x npix box centered on (ra, dec) using the tile's WCS,
    rather than assuming the galaxy sits at a fixed pixel offset."""
    wcs = WCS(header)
    x, y = wcs.all_world2pix(ra, dec, 0)
    x, y = int(round(float(x))), int(round(float(y)))
    half = npix // 2
    y0, y1 = y - half, y - half + npix
    x0, x1 = x - half, x - half + npix
    if y0 < 0 or x0 < 0 or y1 > data.shape[0] or x1 > data.shape[1]:
        return None  # galaxy too close to this tile's edge for a full crop
    return data[y0:y1, x0:x1]

In [4]:
# ------------------------------------------------------------------
# Module-level (picklable) worker function for ProcessPoolExecutor.
# Must NOT close over catalog arrays -- everything it needs comes in
# through `args`, since each call gets pickled and sent to a separate
# process. Returns None on any failure so the caller can filter it.
# ------------------------------------------------------------------
def read_one_worker(args):
    (i, path_str, anchor_source, old_anchors, npix,
     ra_i, dec_i, main_type_i, targetid_i) = args
 
    path = Path(path_str)
    t_start = time.perf_counter()
    img_data = None
 
    try:
        if old_anchors and anchor_source == 1:
            if not path.exists():
                return None
            with fits.open(
                path,
                memmap=True,
                ignore_missing_simple=True,
                ignore_missing_end=True
            ) as hdul:
                img_data = hdul[0].data.copy()
 
        else:
            directory = path
            if not directory.exists():
                return None
 
            prefix = directory.name.strip()
            base = f"SGA2025_{prefix}"
            bands = {}
            for band in ("g", "r", "z"):
                p = directory / f"{base}-image-{band}.fits.fz"
                try:
                    data, header = fitsio.read(p, header=True)
                    cropped = crop_at_radec(data, header, ra_i, dec_i, npix)
                    if cropped is None:
                        continue  # skip this band; get_band's fallback picks another band
                    bands[band] = cropped
                except OSError:
                    continue
 
            if len(bands) < 3:
                p = directory / f"{base}-image-i.fits.fz"
                try:
                    bands["i"] = fitsio.read(p)[39:191, 39:191] # Incorrectly cropped!
                except OSError:
                    pass
 
            if len(bands) < 1:
                return None
 
            g = get_band(bands, ["g", "r", "i", "z"])
            r = get_band(bands, ["r", "i", "g", "z"])
            z = get_band(bands, ["z", "i", "r", "g"])
 
            if any(b is None for b in [g, r, z]):
                return None
            if not all(b.shape == (npix, npix) for b in [g, r, z]):
                return None
 
            img_data = np.stack([g, r, z], axis=0).astype(np.float32, copy=False)
 
    except Exception as e:
        print(f"Unexpected error reading index {i} ({path}): {e}")
        return None
 
    if img_data is None or img_data.shape != (3, npix, npix):
        return None
 
    return {
        "images": img_data,
        "ra": float(ra_i),
        "dec": float(dec_i),
        "Main_Type": int(main_type_i),
        "targetid": int(targetid_i),
        "anchor_source": int(anchor_source),
        "_read_time": time.perf_counter() - t_start,
    }

In [5]:
def create_new_h5(catalog, output_path, verbose, old_anchors=False):
    n_galaxies = len(catalog)
 
    ra = catalog["target_ra"].to_numpy()
    dec = catalog["target_dec"].to_numpy()
    main_type = catalog["Main_Type"].to_numpy(np.int32)
    targetid = catalog["targetid"].to_numpy(np.int64)
    anchor_source = catalog["anchor_source"].to_numpy(np.int8)
    paths = [str(p).strip() for p in catalog["Path"]]
 
    # Precompute the per-galaxy arg tuples once -- these are what get
    # pickled and sent to worker processes, so keep them small/scalar.
    all_args = [
        (i, paths[i], int(anchor_source[i]), old_anchors, npix,
         ra[i], dec[i], main_type[i], targetid[i])
        for i in range(n_galaxies)
    ]
 
    if output_path.exists():
        if verbose:
            print(f"Resuming from {output_path}")
 
        f = h5py.File(output_path, "r+")
        valid_entries = int(f.attrs.get("valid_entries", 0))
        if valid_entries > 0:
            if not np.array_equal(
                f["targetid"][:valid_entries],
                targetid[:valid_entries]
            ):
                raise ValueError("Existing HDF5 does not match this catalog.")
 
    else:
        f = h5py.File(output_path, "w")
        valid_entries = 0
 
    with f:
        datasets = {}
        if valid_entries == 0:
 
            datasets["images"] = f.create_dataset(
                "images",
                (n_galaxies, 3, npix, npix),
                dtype="f4",
                chunks=(batch_size, 3, npix, npix),
            )
 
            for dset_name, dtype in {
                "ra": "f8",
                "dec": "f8",
                "Main_Type": "i4",
                "targetid": "i8",
                "anchor_source": "i4",
            }.items():
                datasets[dset_name] = f.create_dataset(dset_name, (n_galaxies,), dtype=dtype)
            f.flush()
        else:
 
            for dset_name in [
                "images",
                "ra",
                "dec",
                "Main_Type",
                "targetid",
                "anchor_source",
            ]:
                datasets[dset_name] = f[dset_name]
 
        batch = []
        n_failed = 0
        read_times = deque(maxlen=5000)   # rolling window, bounded memory over 500k galaxies
        write_times = deque(maxlen=5000)
        
        def write_batch():
            """Flush the current `batch` list of result dicts to the datasets."""
            nonlocal valid_entries, batch
            n = len(batch)
            if n == 0:
                return
            s = slice(valid_entries, valid_entries + n)
            t0 = time.perf_counter()
            images = np.stack([b["images"] for b in batch])
            datasets["images"][s] = images
            datasets["ra"][s] = np.fromiter((b["ra"] for b in batch), np.float64)
            datasets["dec"][s] = np.fromiter((b["dec"] for b in batch), np.float64)
            datasets["Main_Type"][s] = np.fromiter((b["Main_Type"] for b in batch), np.int32)
            datasets["targetid"][s] = np.fromiter((b["targetid"] for b in batch), np.int64)
            datasets["anchor_source"][s] = np.fromiter((b["anchor_source"] for b in batch), np.int8)
            write_times.append(time.perf_counter() - t0)
            valid_entries += n
            f.attrs["valid_entries"] = valid_entries
            pbar.update(n)
            batch = []
 
        with ProcessPoolExecutor(max_workers=workers) as executor:
            with tqdm(total=n_galaxies, initial=valid_entries) as pbar:
 
                for start in range(valid_entries, n_galaxies, chunk_size):
                    end = min(start + chunk_size, n_galaxies)
 
                    for result in executor.map(read_one_worker, all_args[start:end], chunksize=16):
                        if result is None:
                            n_failed += 1
                            pbar.update(1)
                            continue
                        read_times.append(result.pop("_read_time"))
                        batch.append(result)
                        if len(batch) >= batch_size:
                            write_batch()
 
                    # Once per outer chunk_size loop: flush to disk + print stats.
                    # Flushing here instead of every batch_size cuts Lustre syncs
                    # roughly (chunk_size / batch_size)x.
                    f.flush()
                    if read_times:
                        rt = np.array(read_times)
                        wt = np.array(write_times) if write_times else np.array([0.0])
                        gal_per_sec = 1 / rt.mean() * workers  # rough, assumes near-full worker utilization
                        remaining = n_galaxies - valid_entries
                        eta_hours = remaining / max(gal_per_sec, 1e-9) / 3600
                        print(
                            f"read: mean={rt.mean()*1000:.1f}ms p95={np.percentile(rt,95)*1000:.1f}ms | "
                            f"write: mean={wt.mean()*1000:.1f}ms | ~{gal_per_sec:.1f} gal/s | "
                            f"ETA {eta_hours:.2f}h | failed so far={n_failed}"
                        )
 
        # Final partial batch, if any.
        write_batch()
        f.flush()
 
    if verbose:
        print(f"Saved {valid_entries} galaxies to {output_path} ({n_failed} failed/skipped)")
    print(catalog.columns)
    return output_path


def update_existing_h5(catalog, path, output_path, verbose=1):
    
    with h5py.File( path, "r+") as f:

        if len(f["Main_Type"]) != len(catalog):
            raise ValueError("Catalog and HDF5 contain different numbers of galaxies.")
    
        if not np.array_equal(f["targetid"][:], catalog["targetid"].to_numpy(),):
            raise ValueError("targetid ordering does not match.")
    
        f["Main_Type"][:] = catalog["Main_Type"].to_numpy(np.int32)
    
    return path

In [6]:
def get_path(targets, name, run_path, verbose=1):
    dir = run_path / "h5_files"
    dir.mkdir(parents=True, exist_ok=True)
    path = dir / f"{name}.h5"
 
    if iteration == 1:
        path = create_new_h5(targets, path, verbose, old_anchors=OLD_ANCHORS)
    else:
        update_existing_h5(targets, path, verbose)
    return path
 
 
run_path = prepare_output_directory(output_dir, name)
targets = format_data(data_path, anchor_path, data_columns, anchor_columns)
path = get_path(targets, name, run_path, verbose)

  0%|          | 0/5000 [00:00<?, ?it/s]

read: mean=210.8ms p95=341.3ms | write: mean=26.5ms | ~75.9 gal/s | ETA 0.01h | failed so far=0
read: mean=106.8ms p95=294.8ms | write: mean=27.9ms | ~149.8 gal/s | ETA 0.01h | failed so far=0
read: mean=72.1ms p95=256.3ms | write: mean=28.2ms | ~221.8 gal/s | ETA 0.00h | failed so far=0
read: mean=54.8ms p95=236.3ms | write: mean=28.6ms | ~292.0 gal/s | ETA 0.00h | failed so far=0
read: mean=44.4ms p95=227.6ms | write: mean=28.6ms | ~360.4 gal/s | ETA 0.00h | failed so far=0
Saved 5000 galaxies to /pscratch/sd/q/qshimp/Sorter/binary_classifier/sga2025_sample_test3/h5_files/sga2025_sample_test3.h5 (0 failed/skipped)
Index(['target_ra', 'target_dec', 'Main_Type', 'targetid', 'Path',
       'anchor_source'],
      dtype='str')
